# Python Basics for Data Engineering Interviews

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*

This notebook is a hands-on tour of the Python language features that come up in data-engineering interviews.
It is deliberately granular: every idea gets its own explanation and its own small, runnable example, so you can
stop, predict the output, and only then run the cell.

**How to run it**

1. Run the cells top to bottom with `Shift+Enter`, or use *Run All*.
2. Outputs are saved in the notebook, so it reads like a document even without a kernel.
3. Most clusters of examples end with a **Check yourself** cell full of `assert`s; if it runs silently, your
   mental model is correct.

Companion notebooks: `02_python_coding_problems.ipynb` (timed algorithms), `03_sql_interview_questions.ipynb`
(SQL against SQLite), `04_pyspark_concepts.ipynb` (Spark), and `05_interview_questions.ipynb` (mixed bank).

#### A two-line sanity check

Everything here uses the standard library only: no pandas, no numpy, no Spark. Run this cell first; it proves
the kernel is alive and shows which Python you are on. interviewers expect you to know the version you target,
because syntax such as `str | None` and the `zoneinfo` module need Python 3.10+.

In [1]:
import sys

print("python", sys.version.split()[0])
print("ready")

python 3.13.11
ready


## 1. Values, dynamic typing and numbers

### Names are bound to objects

In Python a variable is a *name* bound to an object, and the object carries the type. There is no declaration,
no compile-time type check, and a name can be rebound to an object of a different type at any moment. Data
engineers care because a column that is "usually an int" can arrive as `None` or a string from a messy source,
and the failure shows up at runtime, deep inside a pipeline.

### `type()`, `isinstance()` and numbers

`type(obj)` tells you what an object is, and `isinstance(obj, T)` is the safe runtime check: it respects
subclassing, whereas `type(x) is T` is exact and brittle. Data engineers care because a column that is "usually
an int" can arrive as `None` or a string from a messy source.

`int` is arbitrary precision, so it never overflows. `float` is a 64-bit IEEE-754 double: fine for measurements
and averages, hopeless for exact money. Note that `bool` is a subclass of `int`, which is why `True + True == 2`.

In [2]:
value = 42
print(type(value).__name__)
value = "42"
print(type(value).__name__)
print(isinstance(True, int), type(True) is int)

rows = 10 ** 18 + 7
print(rows, 3 / 4, 1_000_000)       # ints never overflow; _ is cosmetic

int
str
True False
1000000000000000007 0.75 1000000


### True division, floor division, modulo and rounding

`/` always returns a float; `//` floors toward negative infinity; `%` is the remainder and its sign follows the
divisor. Batching code that assumes `/` is integer division, or that splits negative offsets the "obvious" way,
is a classic source of off-by-one bugs when paging through partitions.

`round()` uses banker's rounding (half-to-even) and its second argument is *digits*, not a step, while
`math.floor`/`math.ceil` return ints and are the right tools for bucket boundaries. Interviewers like asking why
`round(2.675, 2)` is not `2.68`.

In [3]:
import math

print(7 / 2, 7 // 2, 7 % 2)
print(-7 // 2, -7 % 2)               # floors toward -inf
print(divmod(7, 2))                  # quotient and remainder
print(round(2.5), round(3.5), round(2.675, 2))
print(math.floor(-2.1), math.ceil(2.1), (95 + 9) // 10)

3.5 3 1
-4 1
(3, 1)
2 4 2.67
-3 3 10


### The float precision trap

Binary floating point cannot represent most decimal fractions exactly, so `0.1 + 0.2 != 0.3`. Never compare
floats with `==` in validation code; use `math.isclose` with an explicit tolerance, or don't use floats for
values that must balance.

In [4]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))
print(1.1 + 2.2)

0.30000000000000004
False
True
3.3000000000000003


### `Decimal` for money

`decimal.Decimal` is exact decimal arithmetic with configurable rounding, and it is the standard answer to
"how do you sum revenue without losing cents?". Build it from *strings*, not floats -- `Decimal(0.1)` inherits
the binary error, while `Decimal("0.1")` is exactly one tenth.

In [5]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2"))     # exact
print(Decimal(0.1))                        # float error carried in

total = sum(Decimal(amount) for amount in ["19.99", "4.01", "0.99"])
print("total:", total)

0.3
0.1000000000000000055511151231257827021181583404541015625
total: 24.99


**Check yourself.** These asserts cover types, division, rounding and money arithmetic. Predict each
value before you run the cell; the asserts are the feedback loop that replaces a code reviewer.

In [6]:
import math
from decimal import Decimal

assert isinstance(7 // 2, int) and isinstance(7 / 2, float)
assert divmod(7, 2) == (3, 1)
assert math.ceil(95 / 10) == 10
assert Decimal("0.1") + Decimal("0.2") == Decimal("0.3")
assert not math.isclose(0.1, 0.2) and math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)
print("section 1 checks passed")

section 1 checks passed


## 2. Strings

### Literals, indexing and immutability

A `str` is an immutable sequence of Unicode code points. Immutability means every "modification" builds a new
string, which is why repeated `+=` in a loop is quadratic and worth flagging in an interview. Indexing is
zero-based and negative indices count from the end.

In [7]:
text = "data"
print(text[0], text[-1], len(text))
print(text[0:2], text[:2], text[2:])

upper = text.upper()
print(upper, text)          # original is untouched: strings are immutable

d a 4
da da ta
DATA data


### f-strings

f-strings are the modern, readable way to build messages, and they support format specs, conversions and even
inline expressions. In logging and exception messages they are strongly preferred over `%` and `.format`;
the only caveat is to keep expressions short and side-effect free.

In [8]:
table, rows, rate = "orders", 1234, 0.98765

print(f"{table}: {rows:,} rows")
print(f"success rate {rate:.1%}")
print(f"{table!r} padded -> |{table:>10}|")
print(f"{rows * 2 = }")               # debug form, Python 3.8+

orders: 1,234 rows
success rate 98.8%
'orders' padded -> |    orders|
rows * 2 = 2468


### Common methods

`strip`, `lower`, `startswith`, `replace` and friends are the workhorses of cleaning raw text. They all return
new strings, so chaining is idiomatic and safe. Remember `strip()` removes whitespace from both ends, and
`strip("ab")` removes any of the characters `a`/`b`, not the substring `"ab"`.

In [9]:
raw = "  Data Engineering  "
print("[" + raw.strip() + "]")
print(raw.strip().lower())
print(raw.strip().replace(" ", "_"))
print("orders_2026.csv".startswith("orders"), end=" ")
print("orders_2026.csv".endswith((".csv", ".json")))

[Data Engineering]
data engineering
Data_Engineering
True True


### `split` and `join` versus `+` in a loop

`split` turns text into a list, `join` turns an iterable of strings back into one string, and `join` is the
only efficient way to concatenate many pieces. `"".join(parts)` allocates once; `total += part` in a loop
allocates a new string every iteration. This is a favourite performance question.

In [10]:
line = "o1,10.5,USD"
fields = line.split(",")
print(fields, len(fields))

row = ["2026-10-04", "o1", "10.5"]
print(",".join(row))
print(" / ".join(["a", "b", "c"]))

['o1', '10.5', 'USD'] 3
2026-10-04,o1,10.5
a / b / c


**Check yourself.** String basics: slicing, immutability, formatting, cleaning and joining.

In [11]:
assert "engineering"[0:3] == "eng"
assert "  x  ".strip() == "x"
assert "-".join(["a", "b"]) == "a-b"
assert "a,b,c".split(",") == ["a", "b", "c"]
assert f"{1234:,}" == "1,234"
assert "Data".lower() == "data" and "Data"[0] == "D"
print("section 2 checks passed")

section 2 checks passed


## 3. Lists

### Literal, indexing and slicing

A `list` is an ordered, mutable sequence of arbitrary objects. Indexing is O(1); slicing copies a range into a
new list. Slicing with a step is how you sample every n-th row, and `[::-1]` reverses. Note that negative step
slices behave differently from `reversed()` for very large lists because they copy.

In [12]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print(numbers, numbers[0], numbers[-1])
print(numbers[2:5], numbers[:3], numbers[-3:])
print(numbers[::2], numbers[::-1])
print(len(numbers), 5 in numbers)

[3, 1, 4, 1, 5, 9, 2, 6] 3 6
[4, 1, 5] [3, 1, 4] [9, 2, 6]
[3, 4, 5, 2] [6, 2, 9, 5, 1, 4, 1, 3]
8 True


### Mutation

`append`, `insert`, `pop`, `remove` and item assignment change the list in place and return `None` -- a common
interview trap is `rows = rows.append(x)`, which sets `rows` to `None`. Mutation is visible through every name
that references the same list, a theme we return to in the mutability section.

In [13]:
rows = ["r1", "r2"]
rows.append("r3")
rows[0] = "r0"
rows.insert(1, "r1.5")
print(rows)

removed = rows.pop()
print(rows, "removed:", removed)
rows.remove("r1.5")
print(rows)

['r0', 'r1.5', 'r2', 'r3']
['r0', 'r1.5', 'r2'] removed: r3
['r0', 'r2']


### `append` vs `extend` vs `+`

`append(x)` adds one element (even if `x` is a list); `extend(iterable)` adds each element; `a + b` builds a
brand-new list and leaves both inputs alone. Getting this wrong produces the infamous list-of-lists bug in
batch collectors, where `append` nests batches instead of flattening them.

In [14]:
batch = [1, 2]
batch.append([3, 4])       # one element: a nested list
print("append:", batch)

batch = [1, 2]
batch.extend([3, 4])       # each element
print("extend:", batch)

combined = [1, 2] + [3, 4]
print("plus  :", combined)

append: [1, 2, [3, 4]]
extend: [1, 2, 3, 4]
plus  : [1, 2, 3, 4]


### Sorting with `key=`

`sorted()` returns a new list and `list.sort()` sorts in place; both accept `key=` so you can sort by a derived
value without touching the data. A key can return a tuple, which is how you implement a deterministic
multi-field sort: primary field ascending, secondary field descending. Ties matter in data pipelines because
non-deterministic ordering breaks reproducible tests.

In [15]:
counts = {"b": 2, "a": 2, "c": 5}
ranked = sorted(counts, key=lambda key: (-counts[key], key))
print(ranked)

records = [{"id": "o2", "amount": 5}, {"id": "o1", "amount": 9}]
by_amount = sorted(records, key=lambda row: row["amount"])
print([row["id"] for row in by_amount])

['c', 'a', 'b']
['o2', 'o1']


**Check yourself.** Lists: slicing, in-place mutation, `append` vs `extend`, and deterministic sorting.

In [16]:
assert [1, 2, 3][::-1] == [3, 2, 1]
assert [1, 2, 3][1:] == [2, 3]

data = [3, 1, 2]
data.sort()
assert data == [1, 2, 3]

nested = [1]
nested.append([2])
assert nested == [1, [2]]
flat = [1]
flat.extend([2])
assert flat == [1, 2]
assert [2, 1] + [3] == [2, 1, 3]
print("section 3 checks passed")

section 3 checks passed


## 4. Tuples

### Immutability is the feature

A tuple is an ordered, immutable sequence. Because it cannot change, a tuple of immutables is hashable and can
be a dict key or a set member -- exactly what you want for a composite key such as `(customer_id, ds)`. The
classic ML mistake of using a list where a hashable key is required is an easy interview warm-up.

In [17]:
record = ("order-1", 42.5)
print(record, record[0], len(record))

try:
    record[0] = "order-2"
except TypeError as exc:
    print("TypeError:", exc)

lookup = {("cust-1", "2026-10-04"): 12}
print(lookup[("cust-1", "2026-10-04")])

('order-1', 42.5) order-1 2
TypeError: 'tuple' object does not support item assignment
12


### Unpacking and swapping

Tuple unpacking binds several names in one statement and works with any iterable. It is the idiomatic Python
swap, the idiomatic way to receive a function's multiple return values, and the reason `for key, value in
items` is so readable. Use `*rest` to collect the remainder.

In [18]:
order_id, amount = ("o1", 10.5)
print(order_id, amount)

left, right = 1, 2
left, right = right, left
print(left, right)

first, *rest = [1, 2, 3, 4]
print(first, rest)
head, *middle, tail = (1, 2, 3, 4)
print(head, middle, tail)

o1 10.5
2 1
1 [2, 3, 4]
1 [2, 3] 4


### Fixed-shape records

The convention is: tuples for heterogeneous fixed-shape records (a row, a key, a return value), lists for
homogeneous variable-length sequences (a batch of rows). A tuple also documents arity -- `(id, amount)` says
exactly two fields will always be there, which is why query results and coordinates are usually tuples.

In [19]:
Point = tuple  # for illustration only
point = (3, 4)
x, y = point
print("manhattan:", abs(x) + abs(y))

def min_max(values):
    return min(values), max(values)

low, high = min_max([4, 1, 9])
print(low, high)

manhattan: 7
1 9


**Check yourself.** Tuples: immutability, hashability, unpacking and swapping.

In [20]:
assert (1, 2) + (3,) == (1, 2, 3)
assert (1, 2)[0] == 1
a, b = 1, 2
a, b = b, a
assert (a, b) == (2, 1)
assert {("k", 1): "v"}[("k", 1)] == "v"
assert min_max([2, 8]) == (2, 8)
print("section 4 checks passed")

section 4 checks passed


## 5. Sets

### Deduplication and membership

A `set` stores unique, hashable elements with average O(1) membership tests, versus O(n) for a list. That one
fact answers a large share of "how would you speed this up" questions: build a set of seen IDs once, then test
membership in a loop. Sets are unordered, so never rely on their iteration order.

In [21]:
events = ["click", "view", "click", "buy", "view"]
print("list:", events)
print("set :", set(events))
print("unique count:", len(set(events)))

seen = {"a", "b"}
print("a" in seen, "z" in seen)         # O(1) average

list: ['click', 'view', 'click', 'buy', 'view']
set : {'view', 'buy', 'click'}
unique count: 3
True False


### Set algebra

`|` is union, `&` is intersection, `-` is difference and `^` is symmetric difference; each also has a named
method and an in-place operator (`|=`, `&=`, ...). Set algebra is the clearest way to express reconciliation
logic: rows only in yesterday's file, rows in both, rows that appeared or disappeared.

In [22]:
yesterday = {"o1", "o2", "o3"}
today = {"o2", "o3", "o4"}

print("union       :", sorted(yesterday | today))
print("intersection:", sorted(yesterday & today))
print("only today  :", sorted(today - yesterday))
print("symmetric   :", sorted(yesterday ^ today))

union       : ['o1', 'o2', 'o3', 'o4']
intersection: ['o2', 'o3']
only today  : ['o4']
symmetric   : ['o1', 'o4']


**Check yourself.** Sets: dedupe, membership and the four set operations.

In [23]:
assert len(set([1, 1, 2, 3])) == 3
assert set("mississippi") == set("mississippi")
assert {1, 2} | {3} == {1, 2, 3}
assert {1, 2, 3} & {2, 3, 4} == {2, 3}
assert {1, 2, 3} - {2} == {1, 3}
assert {1, 2} ^ {2, 3} == {1, 3}
assert 3 in {1, 2, 3}
print("section 5 checks passed")

section 5 checks passed


## 6. Dictionaries

### Literal access versus `get`

A dict maps hashable keys to values with average O(1) lookup, and since Python 3.7 it preserves insertion order.
`d[key]` raises `KeyError` when the key is missing; `d.get(key, default)` returns a fallback. In pipeline code
`get` is usually what you want for optional columns, while `[]` is the right choice when a missing key really is
a bug that should fail loudly.

In [24]:
config = {"mode": "append", "compression": "zstd"}
print(config["mode"])
print(config.get("partition_by", "none"))

try:
    config["missing"]
except KeyError as exc:
    print("KeyError:", exc)

append
none
KeyError: 'missing'


### Grouping and counting

`d.setdefault(key, [])` returns the existing value or inserts the default and returns it, which turns a
group-by into a two-line loop. `collections.defaultdict` does the same with less noise, and `collections.Counter`
is a purpose-built tally with helpers such as `most_common`. Both are standard library and both come up in
"solve this without pandas" exercises.

In [25]:
from collections import Counter, defaultdict

orders = [("us", 10), ("eu", 5), ("us", 7), ("apac", 2)]
by_region = {}
for region, amount in orders:
    by_region.setdefault(region, []).append(amount)
print(by_region, {region: sum(v) for region, v in by_region.items()})

counts = defaultdict(int)
for word in "the cat the hat".split():
    counts[word] += 1
print(dict(counts), Counter("mississippi").most_common(2))

{'us': [10, 7], 'eu': [5], 'apac': [2]} {'us': 17, 'eu': 5, 'apac': 2}
{'the': 2, 'cat': 1, 'hat': 1} [('i', 4), ('s', 4)]


### Insertion order and iterating items

Iteration yields keys by default; use `.items()` for pairs, `.values()` for values, and `.keys()` (or the dict
itself) for keys. Because order is insertion order, `list(d.items())` is deterministic -- but if you need a
guaranteed order independent of insertion, sort explicitly.

In [26]:
pipeline = {"extract": "s3", "transform": "spark", "load": "snowflake"}
for stage, tool in pipeline.items():
    print(f"{stage:>10} -> {tool}")

print(list(pipeline))
print(sorted(pipeline.items()))

   extract -> s3
 transform -> spark
      load -> snowflake
['extract', 'transform', 'load']
[('extract', 's3'), ('load', 'snowflake'), ('transform', 'spark')]


### Dict comprehensions

A dict comprehension builds a mapping in one expression and reads like the set-builder notation it is named
after. Keep them to a single `for` and at most one `if`; beyond that a loop is clearer, which is itself a
reasonable thing to say in an interview.

In [27]:
squares = {n: n * n for n in range(5)}
print(squares)

raw = {"a": "1", "b": "2", "skip": ""}
clean = {key: int(value) for key, value in raw.items() if value}
print(clean)

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
{'a': 1, 'b': 2}


**Check yourself.** Dicts: access, defaults, grouping, tallies, ordering and comprehensions.

In [28]:
from collections import Counter, defaultdict

assert {"a": 1}.get("b", 0) == 0
grouped = {}
for key, value in [("a", 1), ("a", 2)]:
    grouped.setdefault(key, []).append(value)
assert grouped == {"a": [1, 2]}

assert dict(defaultdict(int, {"a": 1})) == {"a": 1}
assert Counter("aab").most_common(1) == [("a", 2)]
assert list({"x": 1, "y": 2}.items()) == [("x", 1), ("y", 2)]
assert {n: n for n in range(3)} == {0: 0, 1: 1, 2: 2}
print("section 6 checks passed")

section 6 checks passed


## 7. Mutability, references and copies

### Names reference objects

Assignment never copies: it binds another name to the *same* object, and `id()` exposes that identity. This is
why passing a list into a function and mutating it is visible to the caller, and why two "separate" config
dicts can secretly be one object. Almost every surprising "my data changed" bug traces back to aliasing.

In [29]:
a = [1, 2, 3]
b = a                      # alias, not a copy
b.append(4)
print(a, b, a is b, id(a) == id(b))

c = list(a)                # real copy
c.append(99)
print(a, c, a is c)

[1, 2, 3, 4] [1, 2, 3, 4] True True
[1, 2, 3, 4] [1, 2, 3, 4, 99] False


### Shallow versus deep copy

`copy.copy` creates a new outer container but shares the inner objects; `copy.deepcopy` recursively copies
everything. Shallow copies are cheap and usually enough when you only replace top-level entries; deep copies
protect you when nested structures are mutated, at the cost of memory and time.

In [30]:
import copy

nested = [[1], [2]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)

nested[0].append("boom")
print("shallow:", shallow)     # inner list shared
print("deep   :", deep)        # fully independent

shallow: [[1, 'boom'], [2]]
deep   : [[1], [2]]


### Mutation through function arguments

Python is call-by-object-sharing: the function receives the same object, so mutating it affects the caller,
but rebinding the parameter inside the function does not. Interviewers ask this to see whether you can explain
why a helper that "resets" a list argument silently fails.

In [31]:
def add_row_bad(rows):
    rows = rows + ["new"]      # rebinds locally: caller never sees it

def add_row_good(rows):
    rows.append("new")         # mutates the shared object

rows = ["r1"]
add_row_bad(rows)
print("after bad :", rows)
add_row_good(rows)
print("after good:", rows)

after bad : ['r1']
after good: ['r1', 'new']


**Check yourself.** Aliasing, copies and mutation through arguments.

In [32]:
original = [1, 2, 3]
alias = original
assert alias is original  # two names, one object

clone = original[:]
clone.append(4)
assert original == [1, 2, 3] and clone == [1, 2, 3, 4]

In [33]:
import copy

nested = [[1]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)
nested[0].append(2)
assert shallow[0] == [1, 2]   # shallow shares the inner list
assert deep[0] == [1]         # deep is fully independent

In [34]:
def mutate(items):
    items.append("x")

bucket = []
mutate(bucket)
assert bucket == ["x"]        # the caller's list changed
print("section 7 checks passed")

section 7 checks passed


## 8. Control flow

### `if` / `elif` / `else`

Conditions are expressions; there is no `switch` requirement and no braces, just indentation. In data code the
common shape is a small validation ladder: reject bad input early, classify the row, otherwise continue. Keep
branches shallow and return early rather than nesting -- a code-review point that also reduces cyclomatic
complexity.

In [35]:
def classify(amount):
    if amount < 0:
        return "refund"
    elif amount == 0:
        return "empty"
    else:
        return "sale"

print([classify(value) for value in (-5, 0, 12)])

['refund', 'empty', 'sale']


### Truthiness

Every object has a boolean value: empty containers, `0`, `0.0`, `""`, `None` and `False` are falsy, everything
else is truthy. `if rows:` is idiomatic and safe, but it conflates "empty" with "missing" -- if that distinction
matters, write `if rows is None` explicitly. This is a favourite interview nuance.

In [36]:
for value in [0, 0.0, "", [], {}, None, False, 1, "x", [0]]:
    print(repr(value), "->", bool(value))

0 -> False
0.0 -> False
'' -> False
[] -> False
{} -> False
None -> False
False -> False
1 -> True
'x' -> True
[0] -> True


### Loop control: `for`/`else`, `while`, `break`, `continue`

The `else` on a loop runs only if the loop finished without `break`; it is the clean way to express "search, and
report if not found" without a flag variable. `while` repeats until a condition changes, which suits retry and
cursor loops, and `break`/`continue` exit or skip the current iteration. Always make sure the loop variable
actually changes, or you have written an infinite loop.

In [37]:
for row in [1, 2, 3]:
    if row == 2:
        print("found 2")
        break
else:
    print("no 2")

for row in [1, 3]:
    if row == 2:
        break
else:
    print("no 2 in second loop")

attempt = 0
while True:
    attempt += 1
    if attempt < 3:
        continue
    print("succeeded on attempt", attempt)
    break

found 2
no 2 in second loop
succeeded on attempt 3


### `match` (structural pattern matching, 3.10+)

`match` compares a subject against patterns and can destructure sequences, mappings and classes. It is not a
faster `if`; it shines when routing on shapes, such as dispatching on a config dict or an event type. A `case _
` wildcard is required if you want a default.

In [ ]:
def route(event):
    event_type = event.get("type")
    if event_type == "insert" and "table" in event:
        return f"insert into {event['table']}"
    if event_type == "delete":
        return "delete"
    return "unknown"

print(route({"type": "insert", "table": "orders"}))
print(route({"type": "delete"}))
print(route({"type": "other"}))


**Check yourself.** Branching, truthiness, loop `else`, `break`/`continue` and `match`.

In [39]:
assert bool([]) is False and bool([0]) is True
assert bool(0.0) is False and bool("0") is True

In [40]:
found = False
for row in [1, 2, 3]:
    if row == 3:
        found = True
        break
assert found

total = 0
for row in [1, 2, 3, 4]:
    if row % 2:
        continue
    total += row
assert total == 6

In [ ]:
def route(event):
    event_type = event.get("type")
    if event_type == "insert" and "table" in event:
        return f"insert into {event['table']}"
    return "unknown"

assert route({"type": "insert", "table": "t"}) == "insert into t"
assert route({}) == "unknown"
print("section 8 checks passed")


## 9. Comprehensions and generator expressions

### List comprehensions

A list comprehension expresses "build a list by transforming/filtering an iterable" in one readable expression,
usually faster than an equivalent `for` + `append`. Read it as: output expression, then the loops, then the
conditions. Prefer it when the logic is a single transformation.

In [42]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
evens = [n for n in numbers if n % 2 == 0]
squares = [n * n for n in numbers]
print("evens  :", evens)
print("squares:", squares)

evens  : [4, 2, 6]
squares: [9, 1, 16, 1, 25, 81, 4, 36]


### Dict and set comprehensions

The same syntax builds dicts (with `key: value`) and sets (deduplicating automatically). A set comprehension
is the fastest way to collect distinct partition keys while transforming them at the same time.

In [43]:
words = ["Orders", "orders", "CUSTOMERS", "customers"]
print({w.lower() for w in words})              # set comprehension
print({w.lower(): len(w) for w in words})      # dict comprehension

{'orders', 'customers'}
{'orders': 6, 'customers': 9}


### Generator expressions -- and when *not* to use a comprehension

Parentheses instead of brackets create a generator expression: lazy, single-use and O(1) memory, which makes it
the right argument to `sum`, `any`, `all`, `max` and `join`. Comprehensions are for mapping and filtering, not
for side effects, deep branching or multi-level state. If the expression needs more than one `for` and one `if`,
write a plain loop -- "I would keep a loop for clarity" is a strong interview answer.

In [44]:
total = sum(n * n for n in range(1_000_000))     # no million-element list
print(total, any(n > 999_998 for n in range(1_000_000)))
print(", ".join(str(n) for n in range(4)))

pairs = []
for left in range(3):
    for right in range(3):
        if left != right:
            pairs.append((left, right))
print(pairs)

333332833333500000 True
0, 1, 2, 3
[(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]


**Check yourself.** Comprehensions and generator expressions, plus laziness.

In [45]:
assert [n for n in range(5) if n % 2 == 0] == [0, 2, 4]
assert {n % 2 for n in range(5)} == {0, 1}
assert {n: n * n for n in range(3)} == {0: 0, 1: 1, 2: 4}
assert sum(n for n in range(5)) == 10

lazy = (n for n in range(3))
assert next(lazy) == 0
assert list(lazy) == [1, 2]
print("section 9 checks passed")

section 9 checks passed


## 10. Functions

### `def`, `return` and signatures

Functions are the unit of reuse and testing in a pipeline: a pure function that turns rows into rows is trivial
to unit test, while a function that reads S3, transforms and writes is not. State the signature, return a value,
and avoid printing as a side effect in library code.

In [46]:
def add_tax(amount, rate=0.2):
    return round(amount * (1 + rate), 2)

print(add_tax(100))
print(add_tax(100, rate=0.1))
print(add_tax.__name__, add_tax.__defaults__)

120.0
110.0
add_tax (0.2,)


### Default arguments

Defaults are evaluated **once**, at function definition time. That is fine for immutables and dangerous for
containers, which is why `None` sentinels are the standard pattern. The pitfall is subtle enough that it appears
in almost every Python interview and in plenty of real code review.

In [47]:
def append_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(append_bad("a"))
print(append_bad("b"))          # surprising: the default is shared

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

print(append_good("a"), append_good("b"))

['a']
['a', 'b']
['a'] ['b']


### `*args` and `**kwargs`

`*args` collects extra positional arguments into a tuple and `**kwargs` collects extra keyword arguments into a
dict; together, `def wrapper(*args, **kwargs)` is the skeleton of every decorator you will write. `*args` keeps
logging and batch helpers generic, and `**kwargs` lets a wrapper forward options it does not understand to the
function it wraps. Note that parameters written after `*args` are keyword-only.

In [48]:
def total(*values):
    return sum(values)

def load(table, **options):
    return {"table": table, **options}

print(total(), total(1, 2), total(*[1, 2, 3]))
print(load("orders", mode="overwrite", compression="zstd"))
print(load("events", **{"mode": "append"}))

0 3 6
{'table': 'orders', 'mode': 'overwrite', 'compression': 'zstd'}
{'table': 'events', 'mode': 'append'}


### Keyword-only arguments

A bare `*` in the signature marks everything after it as keyword-only, so `save(path, mode="append")` cannot be
called positionally by mistake. This is a cheap API-design win for functions with several boolean-ish options,
and it makes call sites self-documenting.

In [49]:
def write_partition(table, *, mode="append", dry_run=False):
    return f"{table} mode={mode} dry_run={dry_run}"

print(write_partition("orders"))
try:
    write_partition("orders", "overwrite")
except TypeError as exc:
    print("TypeError:", exc)

orders mode=append dry_run=False
TypeError: write_partition() takes 1 positional argument but 2 were given


### Multiple returns and docstrings

Returning a tuple is the Pythonic way to return several values; the caller unpacks it. Docstrings document the
contract and are what `help()` and your IDE show -- write them for anything that is not self-evident, and
remember that tuple returns are exactly why tuples are called "fixed-shape records".

In [50]:
def summarize(rows):
    """Return (count, total) for an iterable of numeric rows."""
    values = list(rows)
    return len(values), sum(values)

count, total = summarize([1, 2, 3])
print(count, total)
print(summarize.__doc__)

3 6
Return (count, total) for an iterable of numeric rows.


**Check yourself.** Defaults, `*args`, `**kwargs`, keyword-only parameters and tuple returns.

In [51]:
def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

assert append_good("a") == ["a"] and append_good("b") == ["b"]

def total(*values):
    return sum(values)

assert total(1, 2, 3) == 6 and total() == 0

def configured(*, mode="append", **options):
    return {"mode": mode, **options}

assert configured(mode="overwrite", x=1) == {"mode": "overwrite", "x": 1}
assert summarize([2, 4]) == (2, 6)
print("section 10 checks passed")

section 10 checks passed


## 11. Scope and closures

### The LEGB rule

Name lookup goes Local, then Enclosing, then Global, then Built-in. Reading an enclosing variable works without
any keyword; *rebinding* it requires `nonlocal` (enclosing) or `global` (module). Understanding LEGB explains
both closure behaviour and the nasty class of bug where a loop variable leaks into a later function call.

In [52]:
greeting = "global"

def outer():
    greeting = "enclosing"
    def inner():
        return greeting          # reads the enclosing binding
    return inner()

print(outer(), greeting)

enclosing global


### Rebinding outer names: `global` and `nonlocal`

Reading an enclosing variable needs no keyword, but *rebinding* it does: `nonlocal` for the nearest enclosing
function scope and `global` for a module-level name. `global` is occasionally needed for module state such as a
counter, but module-level mutable state breaks test isolation and parallel runs. `nonlocal` is the mechanism
behind closures that keep state, such as factory-produced counters.

In [53]:
counter = 0

def bump():
    global counter
    counter += 1

def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

bump()
bump()
first, second = make_counter(), make_counter()
print(counter, [first(), first(), second(), first()])

2 [1, 2, 1, 3]


### Late binding and the default-argument escape hatch

A lambda or inner function that references a loop variable looks up that variable when it is *called*, not when
it is defined -- so a list of lambdas built in a loop all see the final value. Bind the current value with a
default argument (`lambda i=i: i`) or `functools.partial` to freeze it. This is the same "defaults are
evaluated once" rule from the functions section, used deliberately.

In [54]:
callbacks = [lambda: i for i in range(3)]
print([callback() for callback in callbacks])       # all 2

fixed = [lambda i=i: i for i in range(3)]
print([callback() for callback in fixed])           # 0, 1, 2

[2, 2, 2]
[0, 1, 2]


**Check yourself.** Scope, `nonlocal` and late binding.

In [55]:
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter = make_counter()
assert [counter(), counter(), counter()] == [1, 2, 3]

callbacks = [lambda i=i: i for i in range(3)]
assert [callback() for callback in callbacks] == [0, 1, 2]

hits = []
def record(value):
    hits.append(value)
record("x")
assert hits == ["x"]
print("section 11 checks passed")

section 11 checks passed


## 12. Classes

### `__init__` and `self`

A class bundles state and behaviour; an instance is one object with its own state. `self` is simply the
instance passed in as the first argument, and `__init__` is where you attach per-instance attributes. Data
engineers model connections, configs, clients and domain records this way, so fluency here is table stakes.

In [56]:
class Account:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance

account = Account("ada", 100.0)
print(account.owner, account.balance)
print(vars(account))

ada 100.0
{'owner': 'ada', 'balance': 100.0}


### Instance versus class attributes

Attributes defined in the class body are shared by all instances; attributes set on `self` are per instance.
Python resolves `instance.attr` by checking the instance dict first, then the class. Rebinding a class attribute
therefore affects every instance that has not shadowed it -- handy for defaults, dangerous for mutable state.

In [57]:
class Account:
    currency = "USD"                 # class attribute: shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner           # instance attributes
        self.balance = balance

first, second = Account("ada"), Account("grace", 50)
print(first.currency, second.currency, Account.currency)
print("currency" in vars(first))     # read through to the class, not stored

USD USD USD
False


### Instance methods

An instance method takes `self` and reads or mutates that object. `deposit` is the canonical example: it
validates input, changes state, records history and returns the new balance. Keeping validation inside the
method means every caller gets the same rules -- the object protects its own invariants.

In [58]:
class Account:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))
        return self.balance

account = Account("ada", 100)
print(account.deposit(50), account.balance, account.history)

150 150 [('deposit', 50)]


### Class methods

A `@classmethod` receives the class (`cls`), not the instance, and is the idiomatic *alternative constructor*:
`Account.from_statement(...)` reads better than a bare `Account(...)` plus setup, and it works correctly with
subclasses because `cls` is the subclass that was called.

In [59]:
class Account:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance

    @classmethod
    def from_statement(cls, owner, amounts):
        return cls(owner, sum(amounts))

account = Account.from_statement("ada", [100, 50])
print(account.balance, type(account).__name__)

150 Account


### Static methods

A `@staticmethod` takes neither `self` nor `cls`; it is a plain function namespaced under the class because it
is conceptually part of it. `fee_for(amount)` is pure arithmetic, so it needs no instance state, and calling it
on the class documents that.

In [60]:
class Account:
    def __init__(self, owner):
        self.owner = owner

    @staticmethod
    def fee_for(amount):
        return round(amount * 0.01, 2)

print(Account.fee_for(200))
print(Account("ada").fee_for(200))     # reachable via an instance too

2.0
2.0


**Check yourself.** `__init__`, instance attributes, an instance method, a classmethod constructor and a
static helper.

In [61]:
class Account:
    currency = "USD"                      # class attribute, shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner                # instance attributes
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    @classmethod
    def from_statement(cls, owner, amounts):
        return cls(owner, sum(amounts))

    @staticmethod
    def fee_for(amount):
        return round(amount * 0.01, 2)

In [62]:
account = Account.from_statement("ada", [100, 50])
assert account.balance == 150 and account.currency == "USD"
assert account.deposit(50) == 200
assert Account.fee_for(200) == 2.0
assert isinstance(account, Account)
print("section 12 checks passed")

section 12 checks passed


## 13. Inheritance

### `super()`, overriding and the MRO

A subclass inherits attributes and can override methods; `super()` calls the next implementation in the method
resolution order, which is how you extend parent behaviour instead of replacing it. `__init__` is not called
automatically, so you must call `super().__init__(...)` yourself. The MRO is the linearised list of classes
Python searches for an attribute -- `__mro__` shows it, C3 linearisation guarantees a consistent order for
diamonds, and every class ends at `object`. "Which `save` wins?" is answered by reading the MRO left to right.

In [63]:
class Storage:
    def __init__(self, path):
        self.path = path

    def save(self, rows):
        return f"wrote {len(rows)} rows to {self.path}"

class ParquetStorage(Storage):
    def save(self, rows):
        return super().save(rows) + " as Parquet"

print([cls.__name__ for cls in ParquetStorage.__mro__])
print(ParquetStorage("s3://lake").save([1, 2, 3]))

['ParquetStorage', 'Storage', 'object']
wrote 3 rows to s3://lake as Parquet


### Polymorphism

Calling the same method on different subclasses and getting type-appropriate behaviour is polymorphism. It is
the reason a pipeline can iterate over a list of storage backends and call `save` without an `if/elif` chain.
Duck typing means you do not even need a shared base class, only a compatible method.

In [64]:
class Storage:
    def save(self, rows):
        return f"wrote {len(rows)} rows"

class ParquetStorage(Storage):
    def save(self, rows):
        return super().save(rows) + " as Parquet"

class DeltaStorage(Storage):
    def save(self, rows):
        return f"merged {len(rows)} rows (Delta)"

for backend in (ParquetStorage(), DeltaStorage()):
    print(backend.save([1, 2, 3]))

wrote 3 rows as Parquet
merged 3 rows (Delta)


**Check yourself.** Inheritance, `super()`, MRO order and polymorphic dispatch.

In [65]:
class Storage:
    def save(self, rows):
        return f"wrote {len(rows)} rows"

class ParquetStorage(Storage):
    def save(self, rows):
        return super().save(rows) + " as Parquet"

assert ParquetStorage().save([1]) == "wrote 1 rows as Parquet"
assert [c.__name__ for c in ParquetStorage.__mro__] == ["ParquetStorage", "Storage", "object"]
assert isinstance(ParquetStorage(), Storage)

backends = [Storage(), ParquetStorage()]
assert [b.save([1, 2]) for b in backends] == ["wrote 2 rows", "wrote 2 rows as Parquet"]
print("section 13 checks passed")

section 13 checks passed


## 14. Dunder methods

### `__repr__` versus `__str__`

`__repr__` is the unambiguous, developer-facing representation (ideally `eval`-able) used in the REPL, in error
messages and inside containers; `__str__` is the human-friendly form used by `print` and f-strings. If you only
implement one, implement `__repr__`, because `str` falls back to it. Interviewers ask this because debuggable
objects save hours.

In [66]:
class Metric:
    def __init__(self, name, value):
        self.name = name
        self.value = value

    def __repr__(self):
        return f"Metric({self.name!r}, {self.value!r})"

    def __str__(self):
        return f"{self.name}={self.value}"

metric = Metric("rows", 12)
print(metric)             # __str__
print(repr(metric))       # __repr__
print([metric])           # containers always use __repr__

rows=12
Metric('rows', 12)
[Metric('rows', 12)]


### `__eq__`

Defining `__eq__` makes `==` compare your object field by field instead of by identity. Return `NotImplemented`
(not `False`) for unsupported types so Python can try the reflected comparison; that is the difference between a
correct operator and one that silently misbehaves.

In [67]:
class Money:
    def __init__(self, cents, currency="USD"):
        self.cents, self.currency = int(cents), currency

    def __repr__(self):
        return f"Money({self.cents}, {self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.cents, self.currency) == (other.cents, other.currency)

print(Money(100) == Money(100))
print(Money(100) == Money(101))
print(Money(100) == "100")

True
False
False


### `__hash__`

Objects used as dict keys or set members must be hashable, and equal objects must hash equally. Defining
`__eq__` sets `__hash__` to `None` unless you define it too -- so a class that was hashable silently becomes
unhashable. The fix is to hash the same fields you compared.

In [68]:
class Unhashable:
    def __eq__(self, other):
        return True

try:
    hash(Unhashable())
except TypeError as exc:
    print("TypeError:", exc)

class Money:
    def __init__(self, cents):
        self.cents = cents

    def __eq__(self, other):
        return isinstance(other, Money) and self.cents == other.cents

    def __hash__(self):
        return hash(self.cents)

print(len({Money(1), Money(1), Money(2)}))

TypeError: unhashable type: 'Unhashable'
2


### `__add__`

Operator overloading lets domain values behave like numbers: `Money + Money` reads better than
`add_money(a, b)` and keeps currency validation in one place. Raise on incompatible operands, and remember that
`__radd__` (not shown) covers `sum([...])` and reversed operands.

In [69]:
class Money:
    def __init__(self, cents, currency="USD"):
        self.cents, self.currency = int(cents), currency

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")
        return Money(self.cents + other.cents, self.currency)

    def __repr__(self):
        return f"Money({self.cents}, {self.currency!r})"

print(Money(1250) + Money(100))

Money(1350, 'USD')


### Making a container: `__len__`, `__bool__`, `__iter__`, `__getitem__`

`__len__` makes `len(obj)` work and, when `__bool__` is absent, also drives truthiness; defining both is the
clearest contract. `__iter__` makes your object usable in `for` loops and in `list(...)`, `sum(...)` and
`any(...)`, while `__getitem__` enables indexing, slicing and (as a fallback) iteration. Implementing them is
how a `Batch` wrapper around rows starts behaving like the rows themselves.

In [70]:
class Batch:
    def __init__(self, rows):
        self.rows = list(rows)

    def __len__(self):
        return len(self.rows)

    def __bool__(self):
        return bool(self.rows)

    def __iter__(self):
        return iter(self.rows)

    def __getitem__(self, index):
        return self.rows[index]

batch = Batch(["r1", "r2", "r3"])
print(len(batch), bool(Batch([])))
print(list(batch), batch[0], batch[-1], batch[1:])

3 False
['r1', 'r2', 'r3'] r1 r3 ['r2', 'r3']


### Also worth knowing: `__enter__`, `__exit__`, `__call__`

`__enter__`/`__exit__` make `with` work (the next sections use them), and `__call__` makes an instance callable
like a function, which is how you write stateful "functions" or tiny middleware. You do not need to memorise
every dunder; know these plus `__init__`, `__repr__`, `__eq__`, `__hash__`, `__len__` and `__iter__`.

In [71]:
class Doubler:
    def __call__(self, value):
        return value * 2

def apply(fn, values):
    return [fn(value) for value in values]

print(apply(Doubler(), [1, 2, 3]))

[2, 4, 6]


**Check yourself.** `__repr__`/`__str__`, `__eq__`/`__hash__`, `__add__`, `__len__`, `__iter__` and
`__call__`.

In [72]:
class Money:
    def __init__(self, cents):
        self.cents = int(cents)

    def __repr__(self):
        return f"Money({self.cents})"

    def __str__(self):
        return f"${self.cents / 100:.2f}"

    def __eq__(self, other):
        return isinstance(other, Money) and self.cents == other.cents

    def __hash__(self):
        return hash(self.cents)

    def __add__(self, other):
        return Money(self.cents + other.cents)

In [73]:
assert repr(Money(1250)) == "Money(1250)"
assert str(Money(1250)) == "$12.50"
assert Money(1) + Money(2) == Money(3)
assert len({Money(1), Money(1), Money(2)}) == 2   # __eq__ + __hash__ agree
print("section 14 checks passed")

section 14 checks passed


## 15. Properties and encapsulation

### Computed attributes with `@property`

`@property` lets a method be accessed like an attribute, so `temperature.fahrenheit` needs no parentheses and
callers never learn whether the value is stored or derived. Data engineers use this for derived fields
(percentage, duration, partition date) that must stay consistent with the stored state.

In [74]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = float(celsius)

    @property
    def celsius(self):
        return self._celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

temperature = Temperature(25)
print(temperature.celsius, temperature.fahrenheit)

25.0 77.0


### Validation in a setter

A setter runs on every assignment, which is the natural place to enforce invariants: no negative timestamps,
no amounts above a limit, no unknown statuses. Callers keep using plain attribute syntax, so validation is
added without breaking the public API -- that is encapsulation in practice.

In [75]:
class Temperature:
    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

temperature = Temperature()
temperature.celsius = -40
print(temperature.celsius)

try:
    temperature.celsius = -300
except ValueError as exc:
    print("ValueError:", exc)

-40.0
ValueError: below absolute zero


**Check yourself.** `@property` getters, computed attributes and validating setters.

In [76]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

temperature = Temperature(25)
assert temperature.fahrenheit == 77.0
temperature.celsius = -40
assert temperature.fahrenheit == -40.0
print("section 15 checks passed")

section 15 checks passed


## 16. Dataclasses, `NamedTuple` and `Enum`

### `@dataclass`

`@dataclass` generates `__init__`, `__repr__` and `__eq__` from annotated fields, turning a five-line
boilerplate class into three lines. Use it for records, configs and internal DTOs -- anywhere you would
otherwise hand-write a constructor and an equality method.

In [77]:
from dataclasses import dataclass

@dataclass
class Order:
    order_id: str
    amount: float

order = Order("o-1", 42.5)
print(order)
print(order == Order("o-1", 42.5), order == Order("o-2", 42.5))

Order(order_id='o-1', amount=42.5)
True False


### `field(default_factory=...)` and `frozen=True`

Mutable defaults are forbidden in dataclasses for the same reason they are dangerous in functions: one list would
be shared by every instance, so `field(default_factory=list)` asks the constructor to call `list()` per instance.
`frozen=True` makes instances immutable and therefore hashable, so they can be dict keys or set members -- the
dataclass equivalent of `NamedTuple`, and the safe default for value objects such as keys and partition
descriptors.

In [78]:
from dataclasses import dataclass, field

@dataclass
class Cart:
    items: list = field(default_factory=list)

@dataclass(frozen=True)
class Partition:
    table: str
    day: str

first, second = Cart(), Cart()
first.items.append("a")
partition = Partition("orders", "2026-10-04")
print(first.items, second.items)                 # no shared list
print(len({partition, Partition("orders", "2026-10-04")}))

['a'] []
1


### `NamedTuple` and `Enum`

`typing.NamedTuple` gives you a tuple that is also a class: it unpacks positionally like a row, has named fields
for readability, and is immutable and hashable -- choose it when the value genuinely is a tuple with names.
`Enum` replaces magic strings with a closed set of named members, so `status is Status.SHIPPED` is typo-proof for
tooling, and iteration over the enum gives you a canonical list for validation.

In [79]:
from enum import Enum
from typing import NamedTuple

class Summary(NamedTuple):
    partition: str
    rows: int

class Status(Enum):
    PLACED = "placed"
    SHIPPED = "shipped"

summary = Summary("ds=2026-10-04", 120)
print(summary.rows, summary[0], summary.partition)
print([status.value for status in Status], Status("shipped") is Status.SHIPPED)

120 ds=2026-10-04 ds=2026-10-04
['placed', 'shipped'] True


**Check yourself.** Dataclasses, `default_factory`, `frozen=True`, `NamedTuple` and `Enum`. `Status` and
`Summary` are reused from the example above.

In [80]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Order:
    order_id: str
    amount: float
    tags: tuple = ()

@dataclass
class Cart:
    items: list = field(default_factory=list)

In [81]:
order = Order("o-1", 42.5, tags=("priority",))
assert order == Order("o-1", 42.5, tags=("priority",))
assert len({order, Order("o-1", 42.5, tags=("priority",))}) == 1  # frozen => hashable
assert Cart().items == []
assert Cart().items is not Cart().items                          # fresh list per instance
assert Status("placed") is Status.PLACED
assert Summary("ds=1", 5).rows == 5
print("section 16 checks passed")

section 16 checks passed


## 17. Exceptions

### `try` / `except` / `else` / `finally`

`try` runs the risky code, `except` handles a specific failure, `else` runs only when no exception occurred, and
`finally` always runs. Catch the narrowest exception you can actually handle: a bare `except Exception: pass`
hides data bugs. `else` is rare but useful -- it keeps the `try` block to the single operation you are guarding.

In [82]:
def parse_amount(text):
    return float(text)

for candidate in ["10.5", "not-a-number"]:
    try:
        amount = parse_amount(candidate)
    except ValueError as exc:
        print(f"{candidate!r} -> ValueError: {exc}")
    else:
        print(f"{candidate!r} -> parsed {amount}")
    finally:
        print("  done")

'10.5' -> parsed 10.5
  done
'not-a-number' -> ValueError: could not convert string to float: 'not-a-number'
  done


### Custom errors and chaining with `raise ... from`

Define a base exception for your package and subclass it for specific failures: callers can then catch the whole
family or one precise case, and you can attach structured context (the partition name) to the error. `raise New
from original` records the causal chain in `__cause__`, so the traceback says "the above exception was the direct
cause". `raise` with no argument re-raises the active exception. Never swallow the original context; future you
is the one debugging at 3am.

In [83]:
class DataQualityError(Exception):
    """Base class for pipeline data-quality failures."""

class EmptyPartitionError(DataQualityError):
    def __init__(self, partition):
        super().__init__(f"partition {partition} is empty")
        self.partition = partition

try:
    raise EmptyPartitionError("ds=2026-10-03")
except DataQualityError as exc:
    print(type(exc).__name__, "->", exc, "| partition:", exc.partition)

try:
    raise RuntimeError("config unavailable") from OSError("no such file")
except RuntimeError as exc:
    print(type(exc).__name__, "-> cause:", type(exc.__cause__).__name__)

EmptyPartitionError -> partition ds=2026-10-03 is empty | partition: ds=2026-10-03
RuntimeError -> cause: OSError


**Check yourself.** `else`/`finally`, catching a hierarchy, and exception chaining.

In [84]:
class DataQualityError(Exception):
    pass

class EmptyPartitionError(DataQualityError):
    def __init__(self, partition):
        super().__init__(f"partition {partition} is empty")
        self.partition = partition

assert issubclass(EmptyPartitionError, DataQualityError)

In [85]:
steps = []
try:
    raise EmptyPartitionError("ds=2026-10-03")
except DataQualityError as exc:
    steps.append(f"caught {exc.partition}")
else:
    steps.append("no error")
finally:
    steps.append("cleanup")

assert steps == ["caught ds=2026-10-03", "cleanup"]

In [86]:
try:
    raise RuntimeError("outer") from ValueError("inner")
except RuntimeError as exc:
    assert isinstance(exc.__cause__, ValueError)  # chained for the stack trace
print("section 17 checks passed")

section 17 checks passed


## 18. Context managers

### `with` guarantees teardown

A context manager wraps a block with setup and guaranteed cleanup, even when the block raises. Connecting to a
warehouse, opening a transaction, taking a lock and creating a temporary file are all `with` jobs. The guarantee
is the whole point: no leaked handles when an exception unwinds the stack.

In [87]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "part.txt"
    path.write_text("a\nb\n", encoding="utf-8")
    print(path.read_text(encoding="utf-8").splitlines())

print("temp dir removed:", not path.exists())

['a', 'b']
temp dir removed: True


### Writing one with `__enter__` / `__exit__`

`__enter__` returns the value bound by `as`; `__exit__(exc_type, exc, tb)` runs on the way out and can inspect the
exception. Returning `True` swallows the exception and returning `False` (or `None`) lets it propagate -- for a
transaction manager you want `False` plus a rollback, which is what makes partial writes safe when anything in
the block fails.

In [88]:
class Transaction:
    def __init__(self, name):
        self.name = name
        self.log = []

    def __enter__(self):
        self.log.append(f"begin {self.name}")
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("rollback" if exc_type else f"commit {self.name}")
        return False

tx = Transaction("load-orders")
with tx:
    tx.log.append("insert rows")

failing = Transaction("bad-load")
try:
    with failing:
        raise RuntimeError("warehouse down")
except RuntimeError as exc:
    print("propagated:", exc)
print(tx.log, failing.log)

propagated: warehouse down
['begin load-orders', 'insert rows', 'commit load-orders'] ['begin bad-load', 'rollback']


### `contextlib.contextmanager`

`@contextmanager` turns a generator into a context manager: everything before `yield` is setup, everything after
is teardown, and a `try/finally` around the yield makes the cleanup exception-safe. It is the fastest way to
build timers, log scopes and temporary-resource helpers.

In [89]:
from contextlib import contextmanager
import time

@contextmanager
def timer():
    started = time.perf_counter()
    try:
        yield
    finally:
        print(f"elapsed {time.perf_counter() - started:.4f}s")

with timer():
    sum(range(1000))

elapsed 0.0000s


**Check yourself.** `with`, a class-based manager and a generator-based manager. The `Transaction` class
is reused from the example above, which is safe because notebooks execute top to bottom.

In [90]:
committed = Transaction("load")
with committed:
    committed.log.append("insert")
assert committed.log == ["begin load", "insert", "commit load"]

In [91]:
rolled_back = Transaction("load")
try:
    with rolled_back:
        raise ValueError("bad row")
except ValueError:
    pass
assert rolled_back.log[-1] == "rollback"   # __exit__ saw the exception

In [92]:
from contextlib import contextmanager

@contextmanager
def tagging(tag):
    yield tag.upper()

with tagging("ok") as value:
    assert value == "OK"
print("section 18 checks passed")

section 18 checks passed


## 19. Iterators and generators

### Iterable versus iterator

An *iterable* can produce an iterator (`__iter__`); an *iterator* remembers position and returns the next value
(`__next__`) until it raises `StopIteration`. Lists, dicts, files and sets are iterables; `iter(x)` gives the
iterator. A generator is a compact way to write an iterator: a function containing `yield` suspends its frame at
each value, so local state is preserved and memory stays constant regardless of input size. Knowing the
iterable/iterator distinction is a very common screening question.

In [93]:
rows = ["r1", "r2"]
row_iterator = iter(rows)
print(type(row_iterator).__name__, next(row_iterator), next(row_iterator))
try:
    next(row_iterator)
except StopIteration:
    print("exhausted")

def countdown(start):
    while start > 0:
        yield start
        start -= 1

generator = countdown(3)
print(generator)
print(next(generator), next(generator), next(generator))

list_iterator r1 r2
exhausted
<generator object countdown at 0x10beb5000>
3 2 1


### A lazy pipeline over records

Generators compose: parse yields dicts, filter yields a subset, and batching yields lists. No stage materialises
the whole dataset, so you can process a file larger than memory. This is the shape interviewers want when they
ask you to "stream" a transformation.

In [94]:
def parse_orders(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield {"order_id": order_id, "amount": float(amount)}

def only_positive(orders):
    for order in orders:
        if order["amount"] > 0:
            yield order

raw = ["o1,10.5", "o2,-3", "o3,7"]
print(list(only_positive(parse_orders(raw))))

[{'order_id': 'o1', 'amount': 10.5}, {'order_id': 'o3', 'amount': 7.0}]


### Generators versus lists for memory

A list comprehension builds every element up front; a generator expression builds them one at a time. For a
million rows the difference is megabytes versus bytes, plus lower peak memory in the pipeline. The cost is that
a generator is single-use and you cannot index it or take `len()`.

In [95]:
import sys

as_list = [n * n for n in range(10_000)]
as_generator = (n * n for n in range(10_000))

print("list     :", sys.getsizeof(as_list), "bytes")
print("generator:", sys.getsizeof(as_generator), "bytes")
print("sum equal:", sum(as_list) == sum(as_generator))

list     : 85176 bytes
generator: 200 bytes
sum equal: True


**Check yourself.** Iterators, `yield`, lazy composition and single-use semantics.

In [96]:
def countdown(start):
    while start > 0:
        yield start
        start -= 1

assert list(countdown(3)) == [3, 2, 1]
iterator = iter(countdown(2))
assert next(iterator) == 2 and list(iterator) == [1]

def parse_orders(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield {"order_id": order_id, "amount": float(amount)}

parsed = parse_orders(["o1,1.5"])
assert next(parsed) == {"order_id": "o1", "amount": 1.5}

lazy = (n for n in range(3))
assert next(lazy) == 0
assert list(lazy) == [1, 2]
assert list(lazy) == []            # single use
print("section 19 checks passed")

section 19 checks passed


## 20. Decorators

### A function decorator

A decorator is a callable that takes a function and returns a replacement, so you can wrap cross-cutting
behaviour (timing, logging, retrying, caching) around it. The `*args, **kwargs` wrapper forwards anything, and
`functools.wraps` copies the original metadata so the decorated function still looks like itself.

In [97]:
import functools

def traced(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@traced
def add(left, right):
    return left + right

add(1, 2)
add(3, 4)
print(add(5, 6), add.calls, add.__name__)

11 3 add


### `functools.lru_cache`

`lru_cache` memoises results keyed by the arguments, turning exponential recursion into linear work and making
repeated lookups free. Requirements: arguments must be hashable, and the function must be pure. It is a great
one-line answer for "how would you avoid recomputing this lookup?".

In [98]:
import functools

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())

832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


### A retry decorator

Transient failures -- a dropped socket, a throttled API -- are normal in ingestion, so a `retry(times=...)`
decorator that catches one narrow exception and re-raises on the final attempt is standard practice. Keep the
exception type explicit; retrying every exception turns data bugs into slow data bugs.

In [99]:
import functools

def retry(times=3):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except ConnectionError:
                    wrapper.attempts = attempt
                    if attempt == times:
                        raise
        return wrapper
    return decorator

state = {"fails": 2}

@retry(times=5)
def flaky():
    if state["fails"] > 0:
        state["fails"] -= 1
        raise ConnectionError("temporarily unavailable")
    return "ok"

print(flaky(), flaky.attempts)

ok 2


**Check yourself.** A wrapping decorator, `functools.wraps`, memoisation and retry semantics. `traced` and
`fib` are reused from the examples above.

In [100]:
calls = {"n": 0}

@traced
def increment():
    calls["n"] += 1
    return calls["n"]

assert increment() == 1 and increment.calls == 1
assert increment.__name__ == "increment"

@functools.lru_cache(maxsize=None)
def square(n):
    return n * n

assert square(10) == 100
assert square(10) == 100 and square.cache_info().hits == 1
assert fib(20) == 6765
print("section 20 checks passed")

section 20 checks passed


## 21. Type hints

### Annotations document intent

Annotations attach types to parameters, return values and variables: `def average(values: list[float]) ->
float`. Python stores them in `__annotations__` and otherwise ignores them at runtime. They are documentation
that tooling can read, which is why they belong on public functions and data models.

`Optional[T]` (or `T | None`) says "may be missing", `Iterator[T]` says "lazy stream of T", and `Iterable[T]` is
the permissive input you usually want for a parameter. Being explicit about optionality forces callers to handle
the `None` case, which is the most common source of pipeline crashes.

In [101]:
from typing import Iterator, Optional

def row_count(rows: list[dict]) -> int:
    return len(rows)

def average(values: list[float]) -> Optional[float]:
    return sum(values) / len(values) if values else None

def running_total(values: list[float]) -> Iterator[float]:
    total = 0.0
    for value in values:
        total += value
        yield total

print(row_count([{"id": 1}]), row_count.__annotations__)
print(average([]), average([1, 2, 3]), list(running_total([1, 2, 3])))

1 {'rows': list[dict], 'return': <class 'int'>}
None 2.0 [1.0, 3.0, 6.0]


### Hints are not enforcement

Nothing stops you from passing a string where an `int` is annotated; static checkers such as mypy catch it, the
interpreter does not. For data crossing a trust boundary, add real runtime validation (`isinstance`, a schema
library, or explicit tests). Saying this out loud shows you understand the difference between hints and
contracts.

In [102]:
def double(value: int) -> int:
    return value * 2

print(double(21))
print(double("21"))        # no runtime error: hints are metadata only

42
2121


**Check yourself.** Annotations, optional returns and lazy iterators.

In [103]:
from typing import Iterator, Optional

def average(values: list[float]) -> Optional[float]:
    return sum(values) / len(values) if values else None

def running_total(values: list[float]) -> Iterator[float]:
    total = 0.0
    for value in values:
        total += value
        yield total

assert average([1, 2, 3]) == 2.0
assert average([]) is None
assert list(running_total([1, 2, 3])) == [1.0, 3.0, 6.0]
assert row_count.__annotations__["return"] is int
print("section 21 checks passed")

section 21 checks passed


## 22. Dates and times

### Durations and timezone-aware datetimes

`datetime` is a point in time, `date` is a calendar day, and `timedelta` is a duration you can add and subtract.
Always compute durations as timedeltas and convert to seconds explicitly; string arithmetic on timestamps is a bug
factory, and pipeline scheduling, SLA checks and retention windows are all built from these two types.

A naive datetime has no timezone and silently means different instants in different places, so store UTC, attach
`timezone.utc`, and convert with `astimezone` only for display -- `zoneinfo` gives you DST-correct rules. Note
that `datetime.utcnow()` is deprecated precisely because it returns a naive value.

In [104]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
print(started.isoformat(), "->", finished.isoformat())
print("duration seconds:", (finished - started).total_seconds())

as_new_york = finished.astimezone(ZoneInfo("America/New_York"))
print("new york:", as_new_york.isoformat(), "| offset:", as_new_york.utcoffset())

2026-10-04T12:00:00+00:00 -> 2026-10-04T14:30:00+00:00
duration seconds: 9000.0


new york: 2026-10-04T10:30:00-04:00 | offset: -1 day, 20:00:00


### ISO parsing and formatting

`datetime.fromisoformat` parses ISO-8601 strings (the format every API and warehouse returns) and
`.isoformat()` writes them back. Keep the timezone in the string when you serialise; dropping it is how
pipelines accidentally shift timestamps by hours.

In [105]:
from datetime import datetime, timezone

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
print(parsed, parsed.tzinfo is not None)
print(parsed == datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc))
print(parsed.date().isoformat(), parsed.isoformat())

2026-10-04 12:00:00+00:00 True
True
2026-10-04 2026-10-04T12:00:00+00:00


**Check yourself.** Durations, timezone-aware conversion and ISO round-tripping.

In [106]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
assert (finished - started).total_seconds() == 9000

as_new_york = finished.astimezone(ZoneInfo("America/New_York"))
assert as_new_york.hour == 10 and as_new_york.utcoffset() == timedelta(hours=-4)

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
assert parsed == started and parsed.tzinfo is not None
print("section 22 checks passed")

section 22 checks passed


## 23. Files, `pathlib` and JSON

### `Path` basics

`pathlib.Path` is the modern file API: `/` joins, `.suffix` and `.name` inspect, `.read_text`/`.write_text` read
and write, and it works the same on Windows and POSIX. Data code is full of paths, so fluency here is expected;
`.stem`, `.parent` and `.with_suffix` are the other methods worth knowing.

In [107]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    path.write_text("line\n", encoding="utf-8")
    print(path.name, path.suffix, path.stem, path.parent == Path(tmp))
    print(path.exists(), path.read_text(encoding="utf-8").strip())

orders.ndjson .ndjson orders True
True line


### Writing and reading NDJSON

NDJSON (newline-delimited JSON) is the lingua franca of ingestion: one self-contained JSON object per line,
streamable, append-friendly and impossible to corrupt with a partial last line. Open with an explicit encoding
and write one `json.dumps` per row; read it back by iterating lines and `json.loads`-ing each, ideally combining
that with a generator so the file never has to fit in memory.

In [108]:
import json
import tempfile
from pathlib import Path

orders = [{"order_id": "o1", "amount": 10.5}, {"order_id": "o2", "amount": 7.0}]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    with path.open("w", encoding="utf-8") as handle:
        for order in orders:
            handle.write(json.dumps(order) + "\n")
    loaded = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    print(path.read_text(encoding="utf-8").strip().replace("\n", " | "))
    print(loaded == orders, "total:", sum(row["amount"] for row in loaded))

{"order_id": "o1", "amount": 10.5} | {"order_id": "o2", "amount": 7.0}
True total: 17.5


### `json.dumps` and `json.loads`

`json.dumps` serialises to a string (`json.dump` writes to a handle), `json.loads` parses a string
(`json.load` reads a handle). Useful flags: `sort_keys=True` for stable diffs, `indent=2` for humans and
`default=str` to avoid crashing on datetimes. JSON has no datetime type -- decide a wire format and stick to
it.

In [109]:
import json
from datetime import date

print(json.dumps({"b": 1, "a": 2}, sort_keys=True))
print(json.loads('{"n": 3}')["n"])
print(json.dumps({"day": date(2026, 10, 4)}, default=str))

{"a": 2, "b": 1}
3
{"day": "2026-10-04"}


**Check yourself.** `Path` attributes, an NDJSON round-trip and `json` string helpers.

In [110]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    orders = [{"order_id": "o1", "amount": 10.5}, {"order_id": "o2", "amount": 7.0}]
    with path.open("w", encoding="utf-8") as handle:
        for order in orders:
            handle.write(json.dumps(order) + "\n")

    loaded = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    assert loaded == orders
    assert path.suffix == ".ndjson" and path.name == "orders.ndjson"
    assert sum(row["amount"] for row in loaded) == 17.5
assert json.loads(json.dumps({"a": 1})) == {"a": 1}
print("section 23 checks passed")

section 23 checks passed


## 24. Concurrency

### The GIL, threads, processes and asyncio

The global interpreter lock lets only one thread execute Python bytecode at a time, so threads speed up
**I/O-bound** work (waiting on network or disk releases the lock) but not CPU-bound work. Use
`ProcessPoolExecutor` to bypass the GIL for CPU-bound Python, and `asyncio` when the whole call chain is async.
Spark or another distributed engine is the answer when the data is genuinely large.

In [111]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_partition(partition):
    time.sleep(0.01)                    # pretend this is a network call
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
print(partitions[0], len(partitions))

ds=2026-10-01 5


### A deterministic thread pool

`ThreadPoolExecutor.map` applies a function across an iterable using a bounded pool and returns results in input
order. Five sleeps of 10 ms overlap, so the wall clock is far below the ~50 ms a sequential loop would take.
The demo is deterministic: fixed inputs, no randomness, assertion on the results rather than on timing.

In [112]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_partition(partition):
    time.sleep(0.01)
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(fetch_partition, partitions))
elapsed = time.perf_counter() - started

print(results)
print(f"5 mocked fetches in {elapsed * 1000:.1f} ms (sequential would be ~50 ms)")

['rows for ds=2026-10-01', 'rows for ds=2026-10-02', 'rows for ds=2026-10-03', 'rows for ds=2026-10-04', 'rows for ds=2026-10-05']
5 mocked fetches in 13.5 ms (sequential would be ~50 ms)


An `asyncio` version of the same idea, shown to read rather than run because a notebook already owns an event
loop:

```python
import asyncio

async def fetch(partition):
    await asyncio.sleep(0.01)
    return f"rows for {partition}"

async def main():
    return await asyncio.gather(*(fetch(p) for p in partitions))
```

Rule of thumb: threads for blocking I/O libraries that are not async-aware, asyncio when the whole chain is
async, processes for CPU-heavy Python, and a distributed engine for large-scale parallelism.

In [113]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_partition(partition):
    time.sleep(0.01)
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(fetch_partition, partitions))

assert results[0] == "rows for ds=2026-10-01" and len(results) == 5
assert all(result.startswith("rows for ") for result in results)
print("section 24 checks passed")

section 24 checks passed


## 25. Common pitfalls checklist

### Float equality

`0.1 + 0.2 != 0.3` because binary floats cannot represent those decimals exactly. Compare with `math.isclose`
and an explicit tolerance, and never use `==` on floats in validation or tests unless you control the exact bit
pattern.

In [114]:
import math

print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))
print(math.isclose(1e-12, 0.0, abs_tol=1e-9))

False
True
True


### Building strings in a loop

`total += part` copies the whole accumulated string every iteration, which is quadratic. Collect pieces in a
list and `"".join(...)` once, or use an `io.StringIO` for genuinely large output. Interviewers often ask for
the complexity of both approaches.

In [115]:
parts = [str(value) for value in range(5)]
joined = "-".join(parts)
print(joined)

buffer = []
for value in range(5):
    buffer.append(str(value))
print("|".join(buffer))

0-1-2-3-4
0|1|2|3|4


### Identity, equality and shadowed builtins

`==` compares values through `__eq__`, while `is` compares identity (the same object in memory), so use `is` only
for singletons: `None`, `True`, `False` and sentinels. Small integers and interned strings sometimes make `is`
"work" by accident, which is exactly why relying on it is a bug waiting to happen. Similarly, naming a variable
`list`, `dict`, `id`, `sum` or `type` hides the builtin for the rest of the scope -- use `rows`, `mapping`,
`row_id` or `total` instead.

In [116]:
left, right = [1, 2], [1, 2]
print(left == right, left is right, [] is [])
print(None is None)

rows = [1, 2, 3]          # not `list`
total = sum(rows)         # not `sum`
print(total, max(rows), min(rows), list("ab"), dict(a=1))

True False False
True
6 3 1 ['a', 'b'] {'a': 1}


### Mutable defaults and late binding

Two classics to recite: default arguments are evaluated once at definition time (so a default list is shared
across calls), and closures capture variables by reference (so lambdas built in a loop all see the last value).
The fixes are the `None` sentinel and binding the value with `lambda i=i: i`.

In [117]:
def append_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(append_bad("a"), append_bad("b"))     # shared default

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

print(append_good("a"), append_good("b"))
print([callback() for callback in [lambda: i for i in range(3)]])
print([callback() for callback in [lambda i=i: i for i in range(3)]])

['a', 'b'] ['a', 'b']
['a'] ['b']
[2, 2, 2]
[0, 1, 2]


**Check yourself.** Float comparison, join over concatenation, identity versus equality, and the two
classic mutation traps.

In [118]:
import math

assert not math.isclose(0.1 + 0.2, 0.3, rel_tol=0, abs_tol=0)
assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)
assert "-".join(str(i) for i in range(3)) == "0-1-2"
assert [1, 2] == [1, 2] and [1, 2] is not [1, 2]
assert None is None and True is True

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

assert append_good("a") == ["a"] and append_good("b") == ["b"]
assert [callback() for callback in [lambda: i for i in range(3)]] == [2, 2, 2]
print("section 25 checks passed")

section 25 checks passed


## Interview questions

Answer each one out loud before reading the model answer. The answers are deliberately short; your job is to add
the concrete example from the sections above.

**Q.** What is the difference between a list and a tuple, and when do you prefer each?

**A.** Lists are mutable, variable-length and used for homogeneous sequences you will grow or change, such as a
batch of rows. Tuples are immutable, so they are hashable and can be dict keys or set members, and they describe
fixed-shape records such as `(customer_id, ds)`. Prefer a tuple when the arity is part of the contract or when
you need a composite key, and a list when you need to append. In a query result, tuples signal "row shape" and
lists signal "collection".

**Q.** Explain the mutable default argument trap.

**A.** Default arguments are evaluated once, when the `def` statement executes, so `def f(item, bucket=[])`
creates one list shared by every call. Calling `f("a")` then `f("b")` returns `["a", "b"]`, which surprises
everyone the first time. The fix is `bucket=None` plus `bucket = [] if bucket is None else bucket` inside the
function. Dataclasses have the analogous rule and require `field(default_factory=list)`.

**Q.** What is the difference between `__str__` and `__repr__`?

**A.** `__repr__` is the unambiguous developer representation, ideally something like `Money(1250, 'USD')` that
could be re-evaluated, and it is what the REPL, debuggers and containers show. `__str__` is the human-readable
form used by `print` and f-strings. `__str__` falls back to `__repr__`, so if you implement only one, implement
`__repr__`. A good `__repr__` is one of the highest-value debugging investments in a codebase.

**Q.** Why must `__eq__` and `__hash__` be consistent?

**A.** Dicts and sets find objects by hash first and then confirm with `==`, so equal objects must produce equal
hashes. If they do not, a set can hold two "equal" elements or a dict lookup can miss a key that is present.
Defining `__eq__` sets `__hash__` to `None`, silently making the class unhashable, so you must define `__hash__`
over the same fields. `@dataclass(frozen=True)` gives you both for free.

**Q.** When would you use `@classmethod` over `@staticmethod`?

**A.** A `@classmethod` receives `cls` and is the idiomatic alternative constructor:
`Account.from_statement(owner, amounts)` reads better than a bare constructor plus setup, and it respects
subclasses because `cls` is the called class. A `@staticmethod` receives neither `self` nor `cls`; it is a
namespaced pure helper such as `Account.fee_for(amount)` that belongs to the class conceptually but needs no
state. If it touches the class, use classmethod; if it touches nothing, staticmethod is fine.

**Q.** How do generators reduce memory usage?

**A.** A generator produces values lazily, keeping only the current item and the suspended frame instead of
materialising the whole sequence. That is how you stream a file larger than RAM: parse one line, yield one
record, let the consumer write it, and repeat. The trade-offs are that a generator is single-use, cannot be
indexed or measured with `len()`, and holds the function frame open until exhausted. Composing several
generators gives you a streaming pipeline with bounded memory.

**Q.** What does a context manager guarantee?

**A.** That the teardown code runs when the block exits, whether it finished normally or raised an exception.
That is why connections, transactions, locks and temporary files use `with`: no leaked handles even when an
error unwinds the stack. `__exit__` receives the exception type and value, and returning a truthy value would
swallow the exception -- transaction managers return `False` so failures propagate after a rollback.
`contextlib.contextmanager` builds one from a generator with a `try/finally` around the `yield`.

**Q.** What does `functools.wraps` do, and why does it matter?

**A.** It copies `__name__`, `__doc__`, `__module__`, `__qualname__`, `__dict__` and `__wrapped__` from the
wrapped function onto the wrapper. Without it, a decorated function reports the wrapper's name, which breaks log
lines, documentation, tracebacks and tools that introspect signatures. With it, debuggers and `inspect.signature`
still see the original. It is one line and should be in every decorator you write.

**Q.** Explain the GIL and its practical consequence.

**A.** The global interpreter lock allows only one thread to execute Python bytecode at a time, so threads do not
give you parallel CPU work in pure Python. They still help I/O-bound code because blocking calls release the
lock while they wait, which is why a `ThreadPoolExecutor` speeds up many API or database calls. For CPU-bound
work use `ProcessPoolExecutor`, a C extension that releases the GIL, or a distributed engine such as Spark.
Asyncio is the single-threaded alternative for high-concurrency network I/O.

**Q.** What is the difference between `is` and `==`?

**A.** `==` compares values by calling `__eq__`, so two distinct lists with the same contents are equal. `is`
compares identity, meaning the very same object in memory, and is only appropriate for singletons such as
`None`, `True`, `False` and sentinels. CPython caches small integers and interns some strings, so `is` sometimes
appears to work on them -- that is an implementation detail and a latent bug. Write `if value is None`, never
`if value == None`.

**Q.** Why are type hints not enforcement?

**A.** Annotations are stored in `__annotations__` and ignored at runtime, so `double("21")` runs happily even
though the parameter is annotated `int`. Their value is static analysis with mypy or pyright, editor completion
and documentation that cannot drift from the code. To actually enforce types at a boundary -- an API payload, a
config file, a message from Kafka -- add runtime validation with `isinstance`, a schema library, or explicit
tests. Hints describe intent; they do not create a contract.

**Q.** How do you make an instance immutable?

**A.** The simplest options are `@dataclass(frozen=True)`, which blocks attribute assignment and generates
`__hash__`, or `typing.NamedTuple`, which is immutable by construction. You can also define `__setattr__` to
raise, or expose read-only `@property` attributes backed by private state. Immutability makes values safely
hashable, shareable across threads and safe as dict keys. Remember that `frozen=True` is shallow: a mutable
field such as a list inside a frozen dataclass can still be mutated.

**Q.** What is the difference between a shallow copy and a deep copy?

**A.** A shallow copy (`list(x)`, `x.copy()`, `copy.copy`) creates a new outer container but the inner objects
are shared, so mutating a nested list is visible through both copies. `copy.deepcopy` recursively copies
everything, producing a fully independent structure at the cost of time and memory. Use deep copies for nested
config or record structures you intend to mutate, and shallow copies when you only replace top-level entries.
This is also why `dict(config)` does not protect a nested `config["credentials"]`.

**Q.** `d[key]` versus `d.get(key)` -- and when do `setdefault` or `defaultdict` help?

**A.** `d[key]` raises `KeyError` on a missing key, which is right when the key is required and absence is a
bug; `d.get(key, default)` returns a fallback, which is right for optional configuration. `setdefault(key, [])`
returns the existing list or inserts and returns a new one, turning a group-by into a two-line loop.
`collections.defaultdict(list)` expresses the same intent more cleanly, and `Counter` handles tallying with
`most_common`. All of them rely on average O(1) dict lookup, which is why grouping is cheap.

**Q.** Explain late binding in closures.

**A.** A closure captures the variable, not its value, so a function or lambda looks it up when called. Building
`[lambda: i for i in range(3)]` gives three callables that all return `2`, because the loop finished before any
of them ran. Bind the current value with a default argument, `lambda i=i: i`, or use `functools.partial`. The
same mechanism powers useful patterns such as counters that use `nonlocal`, so learn both the power and the
trap.

**Q.** What does `for ... else` do?

**A.** The `else` clause on a loop runs only if the loop completed without hitting `break`. It is the clean way
to express "search and report if not found" without a boolean flag, for example scanning partitions and
branching when no match exists. Many engineers have never seen it, so it is worth knowing -- and worth using
sparingly, because an explicit flag is sometimes clearer to a reviewer.

**Q.** Why prefer `Decimal` over `float` for money?

**A.** Binary floating point cannot represent most decimal fractions exactly, so cents drift when you sum
thousands of amounts. `decimal.Decimal` performs exact decimal arithmetic with explicit rounding and is the
standard-library answer for currency and any value that must reconcile to the cent. Construct it from strings
(`Decimal("19.99")`) because `Decimal(19.99)` inherits the binary error. Floats remain correct for rates,
measurements and statistics where small error is acceptable.

**Q.** Why does `@dataclass` need `field(default_factory=list)`?

**A.** A dataclass turns a default into a value shared by the `__init__` signature, which for a mutable default
reproduces the shared-list bug. `field(default_factory=list)` tells the generated constructor to call `list()`
for every new instance, so each object gets its own container. The same rule covers `dict`, `set` and any
mutable default. Use the factory for every mutable field and a plain default only for immutables.

**Q.** What does `raise ... from` do?

**A.** It sets the new exception's `__cause__` to the original, so the traceback shows "the above exception was
the direct cause of the following exception". That preserves the debugging context while letting you raise an
exception that fits your API, such as wrapping an `OSError` in a `RuntimeError` that names the missing config
file. A bare `raise` inside an `except` block re-raises the active exception unchanged. Never discard the
original error silently.

**Q.** How does the LEGB rule work, and when do you need `global` or `nonlocal`?

**A.** Name lookup checks the Local scope, then any Enclosing function scope, then the Global module scope, then
Built-ins. Reading an enclosing variable needs no keyword, but rebinding it does: `nonlocal` for an enclosing
function scope and `global` for a module-level name. Without the keyword, `count += 1` creates a new local and
raises `UnboundLocalError` if it is also read first. Prefer returning values or using a mutable container; module
state breaks test isolation and parallel runs.

## Self-assessment checklist

Tick each item only when you can explain it *and* write a small example without looking things up.

- [ ] I can name three differences between a list and a tuple, and say which one is hashable.
- [ ] I can explain why `0.1 + 0.2 != 0.3` and what `math.isclose` does instead.
- [ ] I can write a dict comprehension that filters and transforms in one line.
- [ ] I can group rows by key with `setdefault` and again with `defaultdict`.
- [ ] I can explain aliasing, and the difference between `copy.copy` and `copy.deepcopy`.
- [ ] I can describe what `__repr__`, `__eq__` and `__hash__` each control.
- [ ] I can write a class with `__init__`, a classmethod constructor and a static helper.
- [ ] I can trace the MRO of a small inheritance hierarchy.
- [ ] I can add a validating `@property` setter without breaking the attribute API.
- [ ] I can choose between `@dataclass`, `NamedTuple` and `Enum` for a given value type.
- [ ] I can build a custom exception hierarchy and chain with `raise ... from`.
- [ ] I can write a context manager both as a class and with `@contextmanager`.
- [ ] I can turn a list-based function into a lazy generator pipeline.
- [ ] I can write a decorator that uses `functools.wraps`, and explain what `lru_cache` stores.
- [ ] I can say what type hints do and do not enforce at runtime.
- [ ] I can convert a UTC timestamp to another zone with `zoneinfo` and explain why UTC is stored.
- [ ] I can round-trip NDJSON with `pathlib` and `json` without loading it all into memory.
- [ ] I can explain the GIL and pick threads, processes or asyncio for a given workload.
- [ ] I can list the mutable-default, late-binding, `is`/`==` and float-equality pitfalls from memory.

If any box is unticked, jump back to that section and re-run its **Check yourself** cell.

## Keep practising

* `02_python_coding_problems.ipynb` -- timed coding problems with self-checking tests.
* `03_sql_interview_questions.ipynb` -- SQL window functions against a SQLite sandbox.
* `04_pyspark_concepts.ipynb` -- PySpark concepts, guarded so it runs without Spark installed.
* `05_interview_questions.ipynb` -- a mixed question bank for mock interviews.